# Time Series Caching Verification

In [2]:
import sys, os, json
from datetime import date
sys.path.insert(0, os.path.abspath('../..'))

from apps.agentic.core.utils import set_anthropic_env
from apps.agentic.core.series_cache import SeriesCache

set_anthropic_env(filedir='../../.keys')

## SeriesCache

In [3]:
# Initialize
SeriesCache.initialize()

# Put a test entry
fake_data = {
    "observations": [
        {"date": "2020-01-01", "value": "100.0"},
        {"date": "2020-02-01", "value": "101.5"},
    ]
}

cache_id = await SeriesCache.put(
    source="fred",
    native_id="TEST_SERIES",
    data=fake_data,
    observation_start="2020-01-01",
    observation_end="2020-02-01",
    observation_count=2,
    frequency="Monthly",
    units="Index",
    title="Test Series",
)
print(f"cache_id: {cache_id}")

INFO:     SeriesCache initialized: postgresql://yada@localhost/yada
DEBUG:    SeriesCache: stored fred:TEST_SERIES → 19d593ca-f355-4da6-a764-a3d8d7bf761a
cache_id: 19d593ca-f355-4da6-a764-a3d8d7bf761a


In [5]:
# Fetch by cache_id
by_id = await SeriesCache.get_by_cache_id(cache_id)
assert by_id is not None
print(f"get_by_cache_id: {by_id['native_id']} — {by_id['observation_count']} obs")

# Fetch by native_idassert by_id is not None
by_native = await SeriesCache.get_by_native_id("fred", "TEST_SERIES")
assert by_native is not None
print(f"get_by_native_id: {by_native['cache_id']} — expires {by_native['expires_at'].date()}")

# Confirm same cache_id
assert str(by_native['cache_id']) == cache_id
print("OK — both lookups return the same entry")

get_by_cache_id: TEST_SERIES — 2 obs
get_by_native_id: 19d593ca-f355-4da6-a764-a3d8d7bf761a — expires 2026-05-26
OK — both lookups return the same entry


## SeriesRef

In [6]:
from apps.agentic.core.series_ref import SeriesRef

ref = SeriesRef(source="fred", native_id="UNRATE", cache_id="19d593ca-f355-4da6-a764-a3d8d7bf761a")

json_str = ref.to_json()
print(f"to_json:   {json_str}")

ref2 = SeriesRef.from_json(json_str)
print(f"from_json: {ref2}")

assert ref == ref2
print("OK — round-trip preserved all fields")

to_json:   {"source": "fred", "native_id": "UNRATE", "cache_id": "19d593ca-f355-4da6-a764-a3d8d7bf761a"}
from_json: SeriesRef(source='fred', native_id='UNRATE', cache_id='19d593ca-f355-4da6-a764-a3d8d7bf761a')
OK — round-trip preserved all fields


## CachingFredTool

In [3]:
from apps.agentic.core.series_cache import SeriesCache
from apps.agentic.agents.data.caching_fred_tool import CachingFredTool
from lib.mcp_client import MCPClient, MCPClientConfig
from langchain_mcp_adapters.client import MultiServerMCPClient
from apps.agentic.core.constants import MCP_URL

SeriesCache.initialize()

# Get the real MCP tool
mcp_client = MultiServerMCPClient({"fred": {"transport": "sse", "url": MCP_URL}}) # type: ignore
mcp_tools = await mcp_client.get_tools()
fred_tool = next(t for t in mcp_tools if t.name == "fred_series_observations")

tool = CachingFredTool(wrapped=fred_tool)

INFO:     SeriesCache initialized: postgresql://yada@localhost/yada


In [4]:
# Cache miss — should fetch from MCP and store
result1 = await tool._arun(series_id="APU0000708111")
print(f"Miss result: {result1}")

# Cache hit — should return immediately without MCP call
result2 = await tool._arun(series_id="APU0000708111")
print(f"Hit result:  {result2}")
assert result1 == result2
print("OK — same SeriesRef returned on both calls")

DEBUG:    fred_series_observations: cache miss for fred:APU0000708111 — fetching
DEBUG:    SeriesCache: stored fred:APU0000708111 → c8fdf439-2d8d-4fb6-8171-e78a218c7afd
DEBUG:    fred_series_observations: cached fred:APU0000708111 → c8fdf439-2d8d-4fb6-8171-e78a218c7afd
Miss result: {"source": "fred", "native_id": "APU0000708111", "cache_id": "c8fdf439-2d8d-4fb6-8171-e78a218c7afd"}
DEBUG:    fred_series_observations: cache hit for fred:APU0000708111
Hit result:  {"source": "fred", "native_id": "APU0000708111", "cache_id": "c8fdf439-2d8d-4fb6-8171-e78a218c7afd"}
OK — same SeriesRef returned on both calls
